In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import os

# -----------------------------------------------------------------------------
# CONFIG
# -----------------------------------------------------------------------------
class Config:
    batch_size = 32
    block_size = 256
    max_iters = 5000
    eval_interval = 500
    eval_iters = 200

    learning_rate = 3e-4
    warmup_iters = 500

    n_embd = 512
    n_head = 8
    n_layer = 8
    dropout = 0.2

    weight_decay = 0.1
    grad_clip = 1.0
    grad_accum_steps = 4

    device = 'cuda' if torch.cuda.is_available() else 'cpu'

cfg = Config()
torch.manual_seed(1337)

# -----------------------------------------------------------------------------
# DATA
# -----------------------------------------------------------------------------
file_path = '/kaggle/input/datasets/anders372/tiny-shakespeare-dataset/input.txt'
with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(0, len(data_split) - cfg.block_size - 1, (cfg.batch_size,))
    x = torch.stack([data_split[i:i+cfg.block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+cfg.block_size+1] for i in ix])

    return (
        x.pin_memory().to(cfg.device, non_blocking=True),
        y.pin_memory().to(cfg.device, non_blocking=True)
    )

# -----------------------------------------------------------------------------
# RMSNorm
# -----------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x.norm(dim=-1, keepdim=True)
        return x / (norm + self.eps) * self.scale

# -----------------------------------------------------------------------------
# ATTENTION (FLASH)
# -----------------------------------------------------------------------------
class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)

        q = q.view(B, T, cfg.n_head, C // cfg.n_head).transpose(1, 2)
        k = k.view(B, T, cfg.n_head, C // cfg.n_head).transpose(1, 2)
        v = v.view(B, T, cfg.n_head, C // cfg.n_head).transpose(1, 2)

        out = F.scaled_dot_product_attention(
            q, k, v,
            is_causal=True,
            dropout_p=cfg.dropout if self.training else 0.0
        )

        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.dropout(self.proj(out))

# -----------------------------------------------------------------------------
# FEEDFORWARD
# -----------------------------------------------------------------------------
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(cfg.n_embd, 4 * cfg.n_embd),
            nn.GELU(),
            nn.Linear(4 * cfg.n_embd, cfg.n_embd),
            nn.Dropout(cfg.dropout),
        )

    def forward(self, x):
        return self.net(x)

# -----------------------------------------------------------------------------
# BLOCK
# -----------------------------------------------------------------------------
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = MultiHeadAttention()
        self.ff = FeedForward()
        self.ln1 = RMSNorm(cfg.n_embd)
        self.ln2 = RMSNorm(cfg.n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

# -----------------------------------------------------------------------------
# GPT MODEL
# -----------------------------------------------------------------------------
class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.blocks = nn.Sequential(*[Block() for _ in range(cfg.n_layer)])
        self.ln_f = RMSNorm(cfg.n_embd)
        self.head = nn.Linear(cfg.n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok = self.token_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=cfg.device))

        x = tok + pos
        x = self.blocks(x)
        x = self.ln_f(x)

        logits = self.head(x)

        loss = None
        if targets is not None:
            logits = logits.view(B*T, -1)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -cfg.block_size:]
            logits, _ = self(idx_cond)

            logits = logits[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            idx = torch.cat((idx, next_token), dim=1)

        return idx

# -----------------------------------------------------------------------------
# INIT
# -----------------------------------------------------------------------------
model = GPT().to(cfg.device)
model = torch.compile(model)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.learning_rate,
    weight_decay=cfg.weight_decay
)

scaler = torch.cuda.amp.GradScaler()

# -----------------------------------------------------------------------------
# LR SCHEDULER
# -----------------------------------------------------------------------------
def get_lr(step):
    if step < cfg.warmup_iters:
        return cfg.learning_rate * step / cfg.warmup_iters

    progress = (step - cfg.warmup_iters) / (cfg.max_iters - cfg.warmup_iters)
    return 0.5 * cfg.learning_rate * (1 + math.cos(math.pi * progress))

# -----------------------------------------------------------------------------
# EVAL
# -----------------------------------------------------------------------------
@torch.no_grad()
def estimate_loss():
    model.eval()
    out = {}
    for split in ['train', 'val']:
        losses = torch.zeros(cfg.eval_iters)
        for k in range(cfg.eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# -----------------------------------------------------------------------------
# TRAIN
# -----------------------------------------------------------------------------
best_val = float('inf')

for step in range(cfg.max_iters):

    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    if step % cfg.eval_interval == 0:
        losses = estimate_loss()
        print(f"{step}: train {losses['train']:.4f}, val {losses['val']:.4f}")

        if losses['val'] < best_val:
            best_val = losses['val']
            torch.save(model.state_dict(), "best_model.pt")

    optimizer.zero_grad(set_to_none=True)

    for micro_step in range(cfg.grad_accum_steps):
        xb, yb = get_batch('train')

        with torch.cuda.amp.autocast():
            _, loss = model(xb, yb)
            loss = loss / cfg.grad_accum_steps

        scaler.scale(loss).backward()

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)

    scaler.step(optimizer)
    scaler.update()

# -----------------------------------------------------------------------------
# GENERATE
# -----------------------------------------------------------------------------
model.load_state_dict(torch.load("best_model.pt"))
model.eval()

context = torch.zeros((1, 1), dtype=torch.long, device=cfg.device)

print(decode(
    model.generate(context, max_new_tokens=500, temperature=0.8, top_k=40)[0].tolist()
))

/tmp/ipykernel_55/2255932555.py:202: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
W0416 07:48:10.061000 55 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


0: train 4.1839, val 4.1831


/tmp/ipykernel_55/2255932555.py:255: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/tmp/ipykernel_55/2255932555.py:255: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


500: train 3.0828, val 3.0993
1000: train 2.6257, val 2.6202
1500: train 2.5037, val 2.5001
2000: train 2.4225, val 2.4205
2500: train 2.2998, val 2.3056
3000: train 2.2297, val 2.2393
3500: train 2.1753, val 2.1917
4000: train 2.1476, val 2.1681
4500: train 2.1383, val 2.1627

Thy wibll har wor sow sopost, the our meacin hatimed gror gond
Wellest prave, arve is fow swher leath cofffule
To firs resorentule mave lorotiong sandseme;
And ong nit whild thet som emers nownome,
The the fath mowngh cow theim hid the fereouled.

BUSATIOR:
Thint peichonces oung bricioke but Ind plothe's,
Whar teall im she, wour nof hid on-peltith-perast ar-
Voue hill con sthou ang cof thoursst.


DUCRINGhand ERIUS:
I our'd shal oou you there mavestt ar thim
Srres mint bif ay his the thim sit-be
